In [1]:
# Load libraries
import json
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()
MEDIUM_EMAIL = os.getenv("MEDIUM_EMAIL")
MEDIUM_PASSWORD = os.getenv("MEDIUM_PASSWORD")
SUBSTACK_EMAIL = os.getenv("SUBSTACK_EMAIL")
SUBSTACK_PASSWORD = os.getenv("SUBSTACK_PASSWORD")

In [3]:
# Configure Selenium
service = Service(path='./chromedriver')
driver = webdriver.Chrome(service=service)
driver.maximize_window()
wait = WebDriverWait(driver, 60)  # Set explicit wait time

In [5]:
"""Initialize and return a Selenium WebDriver instance."""
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-infobars")
options.add_argument("--disable-popup-blocking")
options.add_argument("--start-maximized")
#options.add_argument("--headless")

service = Service(path='./chromedriver')  # Adjust if necessary
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)  # Set explicit wait time

In [ ]:
#(/Applications/Google Chrome.app)

In [55]:
def login_to_medium(driver):
    try:
        print("Navigating to Medium login page...")
        driver.get("https://medium.com/m/signin")
        time.sleep(2)

        # Click "Sign in with email"
        print("Clicking 'Sign in with email' button...")
        email_signin = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(., 'Sign in with email')]")))
        email_signin.click()
        time.sleep(2)

        # Input email
        print("Waiting for email input field...")
        email_field = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//input[@type='email']")))
        email_field.send_keys(MEDIUM_EMAIL)

        # Click "Continue" button
        print("Clicking 'Continue' button after entering email...")
        continue_button = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(text(), 'Continue')]")))
        continue_button.click()
        time.sleep(2)

        # Ask for email verification code and wait for manual input
        print("Waiting for user to input email verification code...")
        verification_ok_button = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(text(), 'OK')]")))
        verification_ok_button.click()
        time.sleep(5)

        print("Logged in to Medium successfully.")
    except TimeoutException as e:
        print("Error: Timeout while logging in to Medium.")
        print(f"Details: {str(e)}")
    except NoSuchElementException as e:
        print("Error: Unable to locate an element while logging in to Medium.")
        print(f"Details: {str(e)}")


In [56]:
login_to_medium(driver)

In [69]:
def login_to_substack(driver):
    try:
        print("Navigating to Substack login page...")
        driver.get("https://substack.com/sign-in")
        time.sleep(2)

        # Click "Sign in with password"
        print("Clicking 'Sign in with password' link...")
        password_signin = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[@class='login-option substack-login__login-option']")))
        password_signin.click()
        time.sleep(2)

        # Input email
        print("Waiting for email input field...")
        email_field = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//input[@type='email' and @name='email']")))
        email_field.send_keys(SUBSTACK_EMAIL)

        # Input password
        print("Waiting for password input field...")
        password_field = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//input[@type='password' and @name='password']")))
        password_field.send_keys(SUBSTACK_PASSWORD)

        # Click "Continue" button
        print("Clicking 'Continue' button...")
        continue_button = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(text(), 'Continue')]")))
        continue_button.click()
        time.sleep(5)

        print("Logged in to Substack successfully.")
    except TimeoutException as e:
        print("Error: Timeout while logging in to Substack.")
        print(f"Details: {str(e)}")
    except NoSuchElementException as e:
        print("Error: Unable to locate an element while logging in to Substack.")
        print(f"Details: {str(e)}")


In [70]:
login_to_substack(driver)

## Draft mode

In [87]:
import json
import time
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# Load JSON file
def load_draft_content(json_file):
    base_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
    json_path = os.path.join(base_path, json_file)
    with open(json_path, "r", encoding="utf-8") as file:
        return json.load(file)
# Verify content (title and body are mandatory)
def verify_content(content):
    return bool(content.get("title") and content.get("body"))

# Function to save as draft on Medium
def save_to_medium(driver, title, body, tags=None):
    try:
        print("[Medium] Navigating to Medium 'New Story' page...")
        driver.get("https://medium.com/new-story")
        time.sleep(5)

        # Input title
        print("[Medium] Adding title...")
        title_field = driver.find_element(By.XPATH, "//textarea[@placeholder='Title']")
        title_field.send_keys(title)

        # Input body
        print("[Medium] Adding body...")
        body_field = driver.find_element(By.XPATH, "//div[@role='textbox']")
        body_field.send_keys(body)

        # Optionally, handle tags (if provided)
        if tags:
            print("[Medium] Adding tags...")
            for tag in tags:
                tag_field = driver.find_element(By.XPATH, "//input[@placeholder='Add a tag…']")
                tag_field.send_keys(tag)
                tag_field.send_keys(Keys.RETURN)
                time.sleep(1)

        # Save draft (Medium automatically saves drafts; no "Save" button required)
        print("[Medium] Draft saved as draft (Medium autosaves drafts).")
    except Exception as e:
        print(f"[Medium] Error: {e}")

In [88]:
# Function to save as draft on Substack
def save_to_substack(driver, title, body, tags=None, summary=None, audience="all"):
    try:
        print("[Substack] Navigating to Substack 'New Post' page...")
        driver.get("https://substack.com/publish")
        time.sleep(5)

        # Input title
        print("[Substack] Adding title...")
        title_field = driver.find_element(By.XPATH, "//textarea[@name='title']")
        title_field.send_keys(title)

        # Input body
        print("[Substack] Adding body...")
        body_field = driver.find_element(By.XPATH, "//textarea[@name='body']")
        body_field.send_keys(body)

        # Optionally, handle tags (if provided)
        if tags:
            print("[Substack] Adding tags...")
            for tag in tags:
                # Tags feature may not be exposed in all Substack UIs; this is a placeholder for functionality.
                pass

        # Optionally, handle summary (if provided)
        if summary:
            print("[Substack] Adding summary...")
            summary_field = driver.find_element(By.XPATH, "//textarea[@name='summary']")
            summary_field.send_keys(summary)

        # Audience selection (if implemented in Substack UI)
        print(f"[Substack] Audience set to: {audience}")

        # Save draft
        print("[Substack] Saving draft...")
        save_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Save')]")
        save_button.click()
        time.sleep(2)

        print("[Substack] Draft saved successfully.")
    except Exception as e:
        print(f"[Substack] Error: {e}")


In [89]:
if __name__ == "__main__":
    # Load draft content from JSON
    drafts = load_draft_content("data/draft_content_example.json")

    # Initialize Selenium WebDriver
    driver = webdriver.Chrome()

    try:
        for draft in drafts:
            platform_name = draft.get("platform_name")
            title = draft.get("title")
            body = draft.get("body")
            tags = draft.get("tags")
            summary = draft.get("summary")
            audience = draft.get("audience", "all")

            # Verify mandatory content
            if verify_content(draft):
                if platform_name.lower() == "medium":
                    print("[Medium] Starting draft process for Medium...")
                    save_to_medium(driver, title, body, tags)
                elif platform_name.lower() == "substack":
                    print("[Substack] Starting draft process for Substack...")
                    save_to_substack(driver, title, body, tags, summary, audience)
                else:
                    print(f"[Error] Unknown platform: {platform_name}")
            else:
                print(f"[Error] Invalid content in draft: {draft}")
    finally:
        driver.quit()

[Medium] Starting draft process for Medium...
[Medium] Navigating to Medium 'New Story' page...
[Medium] Adding title...
[Medium] Error: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=132.0.6834.160)
Stacktrace:
0   chromedriver                        0x0000000102c770d4 cxxbridge1$str$ptr + 2600792
1   chromedriver                        0x0000000102c6f9f0 cxxbridge1$str$ptr + 2570356
2   chromedriver                        0x00000001028103d8 cxxbridge1$string$len + 89376
3   chromedriver                        0x00000001027eb87c chromedriver + 129148
4   chromedriver                        0x000000010287bdd0 cxxbridge1$string$len + 530200
5   chromedriver                        0x000000010288ecb4 cxxbridge1$string$len + 607740
6   chromedriver                        0x000000010284934c cxxbridge1$string$len + 322708
7   chromedriver                        0x0000000102849f94 cxxbridge1$string$len + 325852
8   chromedr

In [86]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /Users/juli/uoc/03_Portafolio/git-work/beauty-news/scripts
